# Downstream Devices → Semiconductor Output
### An auditable U.S. industrial-production lead-factor falsification · 19 October 2026

**Abstract.** The AI semiconductor cycle is a compelling industry theme, but can U.S. downstream computer and communications-equipment manufacturing *lead* U.S. semiconductor output? We construct a fixed-weight, bounded growth-gap factor and test its incremental forecasting value with 108 expanding-window, purged three-month-ahead observations. **This is a revised-snapshot macro-production study, not a point-in-time executable or equity-alpha strategy.**

## 1. Research question and preregistered hypotheses
**H1 (mechanism):** downstream device-production acceleration precedes upstream semiconductor manufacturing. **H2 (incrementality):** a lagged downstream-versus-upstream production gap reduces out-of-sample RMSE beyond semiconductor persistence and broad manufacturing controls. **H3 (robustness):** any gain is stable to demand-mix weights, longer publication lags, forecast horizons, regimes and negative controls. Failure is informative: an attractive growth industry does not automatically create forecastable alpha.

## 2. Industry structure and why this could fail
NAICS 3341 (computers/peripherals) and NAICS 3342 (communications equipment) are *downstream* U.S. manufacturing proxies. NAICS 3344 (semiconductors and electronic components) is the upstream target. These are **not** company exposure weights: the real semiconductor supply chain is global and inventory/order timing can be reversed. WSTS Spring 2026 projects approximately **$1.51T** global semiconductor sales in 2026 and **$1.9T** in 2027, driven disproportionately by memory and AI. These are forecasts as of 2026, **not historical backtest features**. Source: https://www.wsts.org/76/103/Global-Semiconductor-Market-Surges-Beyond-15T-2026

## 3. Data hierarchy, license and provenance
Grade A: Federal Reserve Board G.17 industrial-production observations, accessed through FRED. Grade B: WSTS public industry outlook, descriptive only. No C-grade media values enter the model. The 189-row snapshot (2011-01 to 2026-09) was retrieved 2026-10-19; FRED marks the latest G.17 update as **2026-10-16**. The downloaded *history* is latest-revised; no ALFRED vintage reconstruction has been performed. See `source_register.csv` for exact URLs, dates, and restrictions.

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from semiconductor_alpha import load_data, make_panel, walk_forward, metrics, run, FEATURES, moving_block_bootstrap, fixed_holdout_placebo
plt.rcParams['svg.fonttype']='none';plt.rcParams['svg.hashsalt']='signals-to-alpha';ROOT=Path.cwd(); (ROOT/'figures').mkdir(exist_ok=True)
raw=load_data(); raw.shape

(189, 4)

### 3.1 Data integrity: timestamps, coverage, missingness
Every month is required; index levels must be positive. The code asserts no duplicate dates or missing observations. These assertions guard ingestion but **cannot** detect economic series revisions or a source misclassification.

In [2]:
qa=pd.DataFrame({'first':[raw.index.min().strftime('%Y-%m')]*4,'last':[raw.index.max().strftime('%Y-%m')]*4,'n':raw.count().values,'missing':raw.isna().sum().values,'min':raw.min().values,'max':raw.max().values},index=raw.columns)
display(qa.round(4)); display(raw.tail(5))

,first,last,n,missing,min,max
IPG3344S,2011-01,2026-09,189,0,53.6659,198.6335
IPG3341S,2011-01,2026-09,189,0,75.0478,184.0602
IPG3342S,2011-01,2026-09,189,0,70.9743,233.9331
IPMAN,2011-01,2026-09,189,0,79.8587,102.0449


,IPG3344S,IPG3341S,IPG3342S,IPMAN
date,,,,
2026-05-01,184.6946,179.2067,228.5258,98.9825
2026-06-01,188.2611,180.3074,230.2037,99.2705
2026-07-01,192.2137,182.4972,231.3716,99.4308
2026-08-01,194.5212,184.0602,233.0737,99.1776
2026-09-01,198.6335,182.6599,233.9331,99.4361


## 4. Real-economy exploration: four distinct series
Index levels are not directly comparable as physical quantities: all are rebased to 2017=100. The question concerns *growth rates and lead/lag relationships*, not the absolute level difference.

In [3]:
ax=raw.plot(figsize=(11,4),lw=1.7,title='Official U.S. industrial-production index levels (latest-revised snapshot)')
ax.set_ylabel('Index (2017=100)');ax.grid(alpha=.2);plt.tight_layout();plt.savefig('figures/01_levels.svg');plt.close()

![Index levels](figures/01_levels.svg)

### 4.1 Log year-over-year growth
Define `g_j,s = 100 log(IP_j,s / IP_j,s−12)`. Log growth is approximately a percentage change and avoids mechanical trend from the 2017-base index.

In [4]:
growth=100*np.log(raw/raw.shift(12)); display(growth.describe().round(2))
ax=growth.loc['2016':].plot(figsize=(11,4),lw=1.5,title='YoY log output growth (descriptive, not point-in-time)');ax.set_ylabel('log-growth, percentage points');ax.axhline(0,color='gray',lw=.7);ax.grid(alpha=.2);plt.tight_layout();plt.savefig('figures/02_growth.svg');plt.close()

,IPG3344S,IPG3341S,IPG3342S,IPMAN
count,177.00,177.00,177.00,177.00
mean,8.13,5.51,7.57,0.19
std,5.83,11.08,9.47,3.56
min,-3.53,-23.54,-11.33,-21.89
25%,3.71,-1.01,0.53,-0.90
50%,8.12,6.60,6.73,0.34
75%,12.12,13.04,13.04,1.48
max,23.21,28.72,36.58,19.34


![Growth](figures/02_growth.svg)

## 5. Point-in-time construction (conservative lag, **not** true vintages)
At forecast origin month `t`, use **only observations dated t−2 or earlier**. This approximates a two-month release buffer for G.17 data; actual archival revision histories are not available. Predict `y_t = semiconductor YoY(t+3)`. To train at origin t, label for past origin u must mature by t−2: `u+3 <= t−2`, so `u <= t−5`. The dataset itself still contains revisions from 2026; therefore the study is *publication-lag-aware* but **not archival PIT**.

## 6. Fixed, non-optimized factor specification

$$Gap_t=\tanh\left(\frac{0.5g_{computer,t-2}+0.5g_{communications,t-2}-g_{semiconductor,t-2}}{15}\right).$$

The 15-point scale and 50/50 mix are fixed for this pilot rather than selected by returns. The factor is bounded in [−1,1]; its sign indicates downstream growth exceeding (or lagging) upstream growth. Because no company segment mix is available, this is **sector-category mapping**, not firm exposure inference.

In [5]:
panel=make_panel(raw,horizon=3,lag=2,weight=.5)
display(panel[['computer_yoy','communications_yoy','semi_yoy','gap','y']].head(5).round(3));print('Valid formation rows:',len(panel))

,computer_yoy,communications_yoy,semi_yoy,gap,y
formation_month,,,,,
2012-03-01,9.370,1.237,5.073,0.015,11.771
2012-04-01,16.134,-0.188,6.523,0.096,12.170
2012-05-01,18.964,-1.273,8.285,0.037,8.471
2012-06-01,21.641,-2.135,9.524,0.015,10.380
2012-07-01,21.905,-2.845,10.494,-0.064,13.179


Valid formation rows: 172


### 6.1 Factor and target diagnostics
Plotting the factor against the realized future YoY target is *ex-post exploratory visualization*, not a trade or a causal claim.

In [6]:
fig,axes=plt.subplots(2,1,figsize=(11,7),sharex=True)
panel.gap.plot(ax=axes[0],title='Fixed-weight demand-supply growth gap',color='tab:blue');axes[0].axhline(0,color='gray',lw=.8)
panel.y.plot(ax=axes[1],title='Realized semiconductor YoY at t+3',color='tab:orange');axes[1].axhline(0,color='gray',lw=.8)
for ax in axes:ax.grid(alpha=.2)
plt.tight_layout();plt.savefig('figures/03_gap_target.svg');plt.close()

![Gap and target](figures/03_gap_target.svg)

## 7. Forecasting architecture and null baselines
All models use a fixed Ridge penalty α=10 and **train-only** standardization. The benchmark ladder is (1) matured-label historical mean, (2) semiconductor YoY persistence, (3) semiconductor YoY + aggregate manufacturing YoY + semiconductor 3-month momentum, and (4) macro controls plus the bounded gap. The expanding window starts only after at least 60 matured labels. A lower RMSE is required before considering any predictive claim.

In [7]:
raw,panel,oos,stats=run(ROOT,simulations=2000)
display(pd.DataFrame(stats['main']).T[['n','rmse','mae','spearman_time_rank_corr_descriptive']].round(4));print('OOS:',stats['oos_first'],'through',stats['oos_last'])

,n,rmse,mae,spearman_time_rank_corr_descriptive
mean,108.0000,5.2218,4.2661,-0.3926
persistence,108.0000,4.4675,3.5958,0.3465
macro,108.0000,4.7128,3.8080,0.3801
full,108.0000,4.7462,3.8147,0.3733
computer_only,108.0000,4.8084,3.8408,0.4243
communications_only,108.0000,4.9238,3.9221,0.3535
breadth,108.0000,4.7545,3.8169,0.3794
delta_rmse_macro_minus_full,-0.0334,-0.0334,-0.0334,-0.0334
delta_rmse_mean_minus_full,0.4756,0.4756,0.4756,0.4756


OOS: 2017-07-01 through 2026-06-01


### 7.1 Explicitly test incremental forecasting value
The *incremental* comparison is full minus macro, not merely full versus zero or a no-change model. Temporal Spearman correlation is a descriptive cross-time association, **not** cross-sectional stock rank IC.

In [8]:
print('RMSE(macro)-RMSE(full):',round(stats['main']['delta_rmse_macro_minus_full'],4),'pp')
print('RMSE(mean)-RMSE(full):',round(stats['main']['delta_rmse_mean_minus_full'],4),'pp')
display(oos[['actual','mean','persistence','macro','full','n_train']].tail(8).round(2))

RMSE(macro)-RMSE(full): -0.0334 pp
RMSE(mean)-RMSE(full): 0.4756 pp


,actual,mean,persistence,macro,full,n_train
formation_month,,,,,,
2025-11-01,9.83,7.83,10.27,8.55,8.70,160
2025-12-01,8.70,7.86,10.55,6.16,6.26,161
2026-01-01,8.70,7.90,10.96,8.16,8.20,162
2026-02-01,11.67,7.93,10.49,10.29,10.36,163
2026-03-01,14.30,7.95,9.69,10.93,10.97,164
2026-04-01,11.37,7.96,9.00,10.01,10.04,165
2026-05-01,13.95,7.96,8.39,7.94,7.96,166
2026-06-01,18.15,7.97,8.38,6.24,6.20,167


### 7.2 Forecast trajectories: what did the model actually predict?
Each point was produced by fitting on matured labels only. This plot is **not** a cumulative P&L or a backtest of investable assets.

In [9]:
ax=oos[['actual','mean','macro','full']].plot(figsize=(11,4),title='Expanding-window OOS forecasts: future semiconductor YoY',lw=1.4)
ax.set_ylabel('YoY log-growth, percentage points');ax.grid(alpha=.2);plt.tight_layout();plt.savefig('figures/04_oos_forecasts.svg');plt.close()

![OOS forecasts](figures/04_oos_forecasts.svg)

### 7.3 Error dynamics, not just a headline RMSE
A model can beat a benchmark on a few crises and lose most of the time. The rolling 12-observation RMSE comparison reveals that temporal concentration; rolling windows overlap and are descriptive.

In [10]:
rolling=pd.DataFrame({k:(oos[k]-oos.actual).pow(2).rolling(12).mean().pow(.5) for k in ['mean','macro','full']})
ax=rolling.plot(figsize=(11,4),title='Trailing 12-origin OOS RMSE (overlapping)',lw=1.5);ax.set_ylabel('RMSE, percentage points');ax.grid(alpha=.2);plt.tight_layout();plt.savefig('figures/05_rolling_rmse.svg');plt.close()

![Rolling RMSE](figures/05_rolling_rmse.svg)

## 8. Leave-one-input-out and negative controls
No firm-level leave-one-out test is possible: there is **no company-level exposure panel**. Instead, drop the computer proxy, communications proxy, or replace the gap with downstream positive-growth breadth. These are genuine model ablations, not fabricated company robustness tests.

In [11]:
display(pd.DataFrame(stats['main']).T.loc[['macro','full','computer_only','communications_only','breadth'],['n','rmse','mae']].round(4))

,n,rmse,mae
macro,108.0,4.7128,3.8080
full,108.0,4.7462,3.8147
computer_only,108.0,4.8084,3.8408
communications_only,108.0,4.9238,3.9221
breadth,108.0,4.7545,3.8169


## 9. Forecast-horizon and publication-lag sensitivity
Re-run the **entire** purged walk-forward at a six-month target and with a three-month publication lag. The information set and training maturity constraint are rebuilt each time, not just shifted predictions.

In [12]:
display(pd.DataFrame({'3m target / 2m lag':stats['main'],'6m target / 2m lag':stats['horizon_6m'],'3m target / 3m lag':stats['publication_lag_3m']}).loc[['mean','macro','full']].applymap(lambda x:round(x['rmse'],3) if isinstance(x,dict) else x))

/tmp/ipykernel_659/1858022944.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display(pd.DataFrame({'3m target / 2m lag':stats['main'],'6m target / 2m lag':stats['horizon_6m'],'3m target / 3m lag':stats['publication_lag_3m']}).loc[['mean','macro','full']].applymap(lambda x:round(x['rmse'],3) if isinstance(x,dict) else x))


,3m target / 2m lag,6m target / 2m lag,3m target / 3m lag
mean,5.222,5.437,5.289
macro,4.713,5.274,4.772
full,4.746,5.404,4.827


## 10. Regime dependence: before COVID and since 2022
A secular technology transition may create unstable relations. Subperiod estimates have smaller samples and should not be treated as independent replications or as optimized timing rules.

In [13]:
regimes=pd.DataFrame({'pre_2020':{k:stats['pre_2020'][k]['rmse'] for k in ['mean','macro','full']},'post_2022':{k:stats['post_2022'][k]['rmse'] for k in ['mean','macro','full']}})
display(regimes.round(3));print('Pre-2020 n:',stats['pre_2020']['full']['n'],'Post-2022 n:',stats['post_2022']['full']['n'])

,pre_2020,post_2022
mean,6.802,4.453
macro,5.525,3.954
full,5.429,3.961


Pre-2020 n: 30 Post-2022 n: 54


## 11. Moving-block bootstrap: uncertainty in incremental RMSE
Resample consecutive six-month blocks of paired OOS errors 2,000 times. Report the 95% percentile interval for RMSE(macro)−RMSE(full). This retains some serial dependence but **does not** account for revisions, model-selection multiplicity, or a fully nested refit in each bootstrap replicate.

In [14]:
print(json.dumps(stats['block_bootstrap'],indent=2));print('Includes zero:',stats['block_bootstrap']['ci95'][0]<=0<=stats['block_bootstrap']['ci95'][1])

{
  "n": 2000,
  "block": 6,
  "ci95": [
    -0.14896898758369953,
    0.07735334603662827
  ],
  "probability_delta_positive_descriptive": 0.258
}
Includes zero: True


## 12. Block-permutation placebo on a fixed holdout
Fit **once** using labels mature before January 2022, then permute **six-month contiguous blocks** of post-2022 realized labels 2,000 times while holding forecasts fixed. This is a **diagnostic alignment null**, not a valid exact test of the adaptive expanding-window coefficient or a p-value for equity alpha. It assumes approximate block exchangeability, which trends can violate.

In [15]:
print(json.dumps(stats['fixed_holdout_block_placebo'],indent=2))

{
  "n_permutations": 2000,
  "n_holdout": 54,
  "split": "2022-01-01",
  "observed_delta_rmse": 0.006168582345077667,
  "p_one_sided_exploratory": 0.7956021989005497,
  "null_95_interval": [
    -0.07021418508531502,
    0.15596708131339512
  ]
}


## 13. Exposure-mix sensitivity (1,000 perturbations)
Because actual device-specific chip content is unknown, the 50/50 weighting is an analyst prior. Vary the computer share uniformly from 20% to 80% in 1,000 fixed pre-2022 fit / post-2022 holdout tests. **This is sensitivity to category proxy weights, not a firm-revenue exposure estimate.**

In [16]:
print(json.dumps(stats['exposure_weight_perturbation'],indent=2))

{
  "n": 1000,
  "weight_range": [
    0.2,
    0.8
  ],
  "holdout_delta_rmse_median": -0.00953770065037407,
  "delta_rmse_5_95": [
    -0.07727237668862479,
    0.01636366426234579
  ],
  "fraction_positive": 0.384
}


## 14. Additional visual falsification: observed versus predicted
A positive regression slope is insufficient: forecast dispersion, extreme misses and baseline comparison all matter. Use only OOS predictions and show the identity line.

In [17]:
fig,ax=plt.subplots(figsize=(6,5))
ax.scatter(oos.actual,oos.macro,s=18,alpha=.5,label='macro controls');ax.scatter(oos.actual,oos.full,s=18,alpha=.5,label='controls + gap')
lo=min(oos[['actual','macro','full']].min());hi=max(oos[['actual','macro','full']].max());ax.plot([lo,hi],[lo,hi],ls='--',color='gray')
ax.set(xlabel='Realized future semiconductor YoY (pp)',ylabel='Predicted YoY (pp)',title='OOS forecast calibration');ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.savefig('figures/06_calibration.svg');plt.close()

![Calibration](figures/06_calibration.svg)

## 15. Bias and validity audit
**Look-ahead:** a two-month observation lag and purged labels avoid explicit forward-row use, but revised historical G.17 data are *not* true point-in-time vintages. **Survivorship:** the sector aggregates are continuing official series; a historical equity universe is absent. **Selection:** NAICS proxy choices, 3/6-month horizons and Ridge α=10 are researcher decisions, not a preregistered external holdout. **Multiple testing:** exploratory subperiods/weights inflate false-discovery risk. **Trading costs:** no asset returns, bid-ask spreads, borrow, slippage or corporate actions exist here. **Causality:** correlation and forecast error do not identify a demand shock. **Overlapping labels:** time-series dependence undermines naive i.i.d. standard errors.

## 16. Industry growth versus factor profitability
WSTS forecasts are global *revenue* forecasts heavily driven by price-sensitive memory, whereas the target here is **U.S. real industrial production** (volume-like index), not global chip sales or company EPS. Growth can be genuine even when the chosen factor fails; the two claims must not be conflated. The tested downstream series also omit imports, foundry capacity and order books.

In [18]:
verdict='FAILS incremental macro forecast test' if stats['main']['delta_rmse_macro_minus_full']<=0 else 'Exploratory incremental gain, subject to robustness'
print('PRIMARY VERDICT:',verdict)
print('Economic sector outlook: structurally interesting; INVESTABLE EQUITY ALPHA: NOT TESTED')
print('Saved files:',len(list((ROOT/'figures').glob('*.svg'))),'SVG figures,',len(oos),'OOS predictions')

PRIMARY VERDICT: FAILS incremental macro forecast test
Economic sector outlook: structurally interesting; INVESTABLE EQUITY ALPHA: NOT TESTED
Saved files: 6 SVG figures, 108 OOS predictions


## 17. Production-grade next steps
1. Download **historical G.17 releases/ALFRED vintages** with timestamps and recompute each origin from the contemporaneously available observation. 2. Acquire legally reusable WSTS/SIA shipments, semi equipment orders, chip inventory and import volumes; predefine release lags. 3. Build *point-in-time* firm segment/geography exposure from SEC filings and test **future company fundamentals** first. 4. Only after fundamental validation, assemble survivorship-free total returns, sector-neutral rank IC, net long-short spreads and realistic costs. 5. Reserve an untouched forward period and apply multiple-testing corrections.

## 18. Reproduction and audit artifacts
- `semiconductor_alpha.py`: signal, purged OOS, ablations, bootstrap, block permutation, weight sensitivity.
- `data/fred_g17_snapshot_2011_2026.csv`: 189 monthly rows from official G.17/FRED, latest-revised snapshot.
- `data/derived_formation_panel.csv`: model formation features/targets.
- `results/walk_forward_oos.csv`, `results/metrics.json`: exact numeric outputs.
- `source_register.csv`: source URLs, release dates, quality and vintage status.
- `tests/test_semiconductor_alpha.py`: integrity, no-future-feature, purge and deterministic simulation tests.
- `figures/*.svg`: actual executed chart artifacts.

**Verdict:** no robust lead-factor evidence; **no investable equity alpha claimed**. Sources: https://fred.stlouisfed.org/series/IPG3344S and https://www.federalreserve.gov/releases/g17/current/default.htm.